# z719 final gerencial v7: learning_rate 0,02 contra 0,03 en cuatro combinaciones (WF7197)
Francisco Medina Raed. En el barrido del WF7196, `learning_rate = 0.02` fue la única configuración que quedó por encima de la base en las dos medidas, pero con 10 semillas no se distinguía del ruido.

Este notebook prueba si bajar el learning rate de 0,03 a 0,02 mejora **de forma consistente**, en las cuatro combinaciones de `num_leaves` y `min_data_in_leaf` que empataron en el grid del WF7193, con **40 semillas** cada una. Todo lo demás es igual al WF7193.

| num_leaves | min_data_in_leaf | Se prueba con learning rate |
|---|---|---|
| 256 | 128 (la base) | 0,03 y 0,02 |
| 512 | 128 | 0,03 y 0,02 |
| 64 | 512 | 0,03 y 0,02 |
| 128 | 512 | 0,03 y 0,02 |

Son 8 configuraciones y 320 entrenamientos.

**Qué muestra**
- Cada configuración contra la base, con la ganancia individual (árboles fijos), su error, y el semillerío.
- **El efecto del learning rate dentro de cada combinación**: 0,02 contra 0,03 con las mismas semillas.
- **El efecto promedio** en las cuatro combinaciones, con su error.
- Semilleríos de a 10 semillas, grupo por grupo.

**La regla, fijada de antemano:** se entrena y se sube un modelo final con `learning_rate = 0.02` (sobre la base 256 / 128) solo si el efecto promedio supera 0,3 M, supera 2 errores y es positivo en al menos 3 de las 4 combinaciones. La decisión se toma por el efecto promedio y no por la mejor configuración suelta, que siempre sale inflada por azar. Si no cumple, no se sube nada y la entrega sigue siendo la del WF7193.

Es reanudable por configuración. Si ya corriste el WF7197 con solo la base y `learning_rate = 0.02`, esas dos se reutilizan y se agregan las otras seis. Primero conviene `modo <- "test"`.

## Parámetros, paquetes y carpetas

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

modo <- "full"           # "test" = corrida corta para probar que todo anda (no sube a Kaggle)
subir_kaggle <- TRUE     # sube los archivos a Kaggle apenas se generan (necesita el kaggle.json)

PARAM <- list()
PARAM$semilla_primigenia <- 123457     # <- tu semilla primigenia
PARAM$experimento <- 7197
PARAM$comparar_con <- c(7190, 7191, 7192, 7193, 7194, 7195, 7196)           # experimentos anteriores que se muestran en la curva de puntajes
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# --- Feature engineering: cada bloque se prende o se apaga por separado ---
# Cada combinacion de bloques se guarda en su propia carpeta y con su propio nombre en Kaggle,
# asi se pueden correr de a uno y comparar sin pisarse.
PARAM$fe$sacar_ids  <- TRUE     # numero_de_cliente, foto_mes y kmes dejan de ser variables del modelo
PARAM$fe$ratios     <- TRUE     # bloque 2: ratios y sumas dentro del mes (senales de fuga)
PARAM$fe$rankings   <- TRUE     # bloque 1: montos convertidos a ranking por mes (data drifting)
PARAM$fe$tendencias <- TRUE     # bloque 3: tendencia, promedio, max, min y ratio en ventana, sobre las top variables
PARAM$fe$tendencias_top     <- 40
PARAM$fe$tendencias_ventana <- 6
PARAM$fe$ratios2    <- FALSE    # NUEVO: segunda tanda de ratios, con columnas que si tiene el dataset gerencial
PARAM$fe$canaritos  <- FALSE    # NUEVO: descarta las variables que el modelo usa menos que variables de puro azar
PARAM$fe$canaritos_cantidad    <- 50     # cuantas variables de azar se agregan
PARAM$fe$canaritos_semillas    <- 5      # modelos con los que se promedia la importancia
PARAM$fe$canaritos_iteraciones <- 200    # arboles de cada uno de esos modelos
PARAM$fe$canaritos_umbral      <- 0.5    # una variable queda si supera a esta fraccion de los canaritos (0.5 = la mediana)

etiqueta <- paste(c(if (PARAM$fe$sacar_ids) "noid", if (PARAM$fe$rankings) "rank",
                    if (PARAM$fe$ratios) "ratio", if (PARAM$fe$ratios2) "ratio2",
                    if (PARAM$fe$tendencias) "tend", if (PARAM$fe$canaritos) "can"), collapse = "_")
if (etiqueta == "") etiqueta <- "base"

# --- Training strategy: la ganadora del Exp 07, del WF7190 y del WF7191 ---
PARAM$variantes <- list(
  list(nombre = paste0("V3fe_", etiqueta), desde = 202103,
       descripcion = paste0("training 202103-202106, final_train 202103-202107, grid por ganancia, FE: ", etiqueta))
)
PARAM$trainingstrategy$validate     <- 202107
PARAM$trainingstrategy$hasta        <- 202106   # ultimo mes de training
PARAM$trainingstrategy$future       <- 202109
PARAM$trainingstrategy$training_pct <- 1.0      # sin undersampling, como el z719
PARAM$trainingstrategy$positivos    <- c("BAJA+1", "BAJA+2")

# --- Ganancia de la competencia ---
PARAM$ganancia$acierto <- 975000    # por cada BAJA+2 contactado
PARAM$ganancia$error   <- -25000    # por cada BAJA+1 o CONTINUA contactado
PARAM$ganancia$ventana <- 401       # suavizado de la curva de ganancia en validacion (clientes)

# --- Grid search: ahora se elige por GANANCIA en 202107, no por AUC ---
# min_data_in_leaf arranca en 64: con 16 y 32 el WF7190 sobreajusto (V2 se desplomo en Kaggle)
PARAM$grid <- list(
  num_leaves       = c(64, 128, 256, 512, 1024),
  min_data_in_leaf = c(64, 128, 256, 512, 1024, 2048)
)
PARAM$grid$semillas_eval <- 10      # cada combinacion se evalua con 10 semillas y se promedia (la ganancia es ruidosa)

# --- Parametros fijos de LightGBM: identicos al z719 ---
PARAM$lgbm$param_fijos <- list(
  objective = "binary",
  metric = "None",              # la metrica es la ganancia (feval), no el AUC
  first_metric_only = TRUE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  verbosity = -100,
  force_row_wise = TRUE,
  seed = PARAM$semilla_primigenia,
  max_bin = 31,
  learning_rate = 0.03,
  feature_fraction = 0.5,
  num_iterations = 2048,        # lo limita el early stopping
  early_stopping_rounds = 200,
  num_leaves = 64,
  min_data_in_leaf = 128
)

# --- Barrido de hiperparametros: se mueve UN parametro por vez sobre la configuracion base ---
# Base: la ganadora del WF7193 (mismas variables, mismos meses)
PARAM$base <- list(num_leaves = 256, min_data_in_leaf = 128)
PARAM$barrido$semillas <- 40            # semillas con las que se evalua cada configuracion (el WF7196 uso 10)
PARAM$barrido$grupo    <- 10            # ademas se arman semillerios de a 10 semillas, para compararlos entre configuraciones
PARAM$barrido$valores <- list(          # solo la configuracion que quedo en duda en el WF7196
  learning_rate = c(0.02)               # base 0.03; para sumar otro valor: c(0.02, 0.01)
)
# la misma comparacion (learning_rate 0.03 contra 0.02) sobre las otras combinaciones de num_leaves y
# min_data_in_leaf que empataron en el grid del WF7193. Cada combinacion va de a pares: _lr0.03 y _lr0.02
PARAM$barrido$extras <- list(
  "nl512_mdil128_lr0.03" = list(num_leaves = 512, min_data_in_leaf = 128),
  "nl512_mdil128_lr0.02" = list(num_leaves = 512, min_data_in_leaf = 128, learning_rate = 0.02),
  "nl64_mdil512_lr0.03"  = list(num_leaves = 64,  min_data_in_leaf = 512),
  "nl64_mdil512_lr0.02"  = list(num_leaves = 64,  min_data_in_leaf = 512, learning_rate = 0.02),
  "nl128_mdil512_lr0.03" = list(num_leaves = 128, min_data_in_leaf = 512),
  "nl128_mdil512_lr0.02" = list(num_leaves = 128, min_data_in_leaf = 512, learning_rate = 0.02)
)
# regla para entrenar y subir un modelo final con learning_rate = 0.02 (sobre la base 256 / 128):
# el efecto de bajar el learning rate, PROMEDIADO en las cuatro combinaciones, tiene que superar umbral_M millones
# y 2 errores, y ser positivo en al menos 3 de las 4 combinaciones
PARAM$barrido$umbral_M <- 0.3
PARAM$barrido$final_manual <- ""        # para forzar una configuracion aunque no pase la regla: "learning_rate=0.02"

# --- Semillerio: el modelo final se entrena con N semillas y se promedian las probabilidades ---
PARAM$semillerio$cantidad <- 40           # 40 para tener de donde sacar las semillas de la comparacion

# --- Opcional: escalar num_iterations por las filas que agrega 202107 al final_train ---
PARAM$final$escalar_iteraciones <- FALSE

# --- Kaggle: cortes de a 50 alrededor del pico del Exp 07 (unos 1.100 envios) ---
PARAM$kaggle$competencia <- "labo-1-virtual-2026-manager"
PARAM$kaggle$cortes <- seq(800, 1400, by = 50)

if (modo == "test") {
  PARAM$experimento <- paste0(PARAM$experimento, "_test")
  PARAM$grid <- list(num_leaves = c(64, 256), min_data_in_leaf = c(64, 512), semillas_eval = 2)
  PARAM$barrido$semillas <- 4
  PARAM$barrido$grupo <- 2
  PARAM$lgbm$param_fijos$num_iterations <- 60
  PARAM$lgbm$param_fijos$early_stopping_rounds <- 20
  PARAM$semillerio$cantidad <- 4
  subir_kaggle <- FALSE
}

# semillas del semillerio: la primera es la primigenia, el resto se derivan de ella (reproducible)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
PARAM$semillerio$semillas <- c(PARAM$semilla_primigenia,
  setdiff(sample(100000:999999, PARAM$semillerio$cantidad + 5), PARAM$semilla_primigenia)[seq_len(PARAM$semillerio$cantidad - 1)])

for (paquete in c("data.table", "R.utils", "lightgbm", "yaml")) {
  if (!require(paquete, character.only = TRUE)) {
    install.packages(paquete)
    require(paquete, character.only = TRUE)
  }
}

# carpeta del experimento en el bucket de la VM (o en el home si no hay bucket)
base <- path.expand(if (dir.exists("~/buckets/b1")) "~/buckets/b1/exp" else "~/exp")
carpeta_exp <- file.path(base, paste0("WF", PARAM$experimento))
dir.create(carpeta_exp, recursive = TRUE, showWarnings = FALSE)
setwd(carpeta_exp)

# dataset: lo busco donde lo suele dejar la VM; si no esta, lo descargo
candidatos <- path.expand(file.path(c("~/datasets", "~/buckets/b1/datasets", carpeta_exp), PARAM$dataset))
archivo_dataset <- candidatos[file.exists(candidatos)][1]
if (is.na(archivo_dataset)) {
  archivo_dataset <- file.path(carpeta_exp, PARAM$dataset)
  download.file(paste0("https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/", PARAM$dataset),
                archivo_dataset, mode = "wb")
}

message("variante: ", PARAM$variantes[[1]]$nombre, "\ncarpeta: ", carpeta_exp, "\ndataset: ", archivo_dataset,
        "\nsemillas del semillerio: ", paste(PARAM$semillerio$semillas, collapse = ", "))

## Preprocesamiento y feature engineering
Igual que el WF7193.

In [ ]:
dataset <- fread(archivo_dataset)

# ---------- 6.3.1.2 Catastrophe Analysis (identico al z719) ----------
dataset[foto_mes == 202006, internet := NA]
dataset[foto_mes == 202006, mrentabilidad := NA]
dataset[foto_mes == 202006, mrentabilidad_annual := NA]
dataset[foto_mes == 202006, mcomisiones := NA]
dataset[foto_mes == 202006, mactivos_margen := NA]
dataset[foto_mes == 202006, mpasivos_margen := NA]
dataset[foto_mes == 202006, mcuentas_saldo := NA]
dataset[foto_mes == 202006, ctarjeta_visa_transacciones := NA]
dataset[foto_mes == 202006, mtarjeta_visa_consumo := NA]
dataset[foto_mes == 202006, mtarjeta_master_consumo := NA]
dataset[foto_mes == 202006, ccallcenter_transacciones := NA]
dataset[foto_mes == 202006, chomebanking_transacciones := NA]

# ---------- 6.3.1.3 FE intra-mes (identico al z719) ----------
atributos_presentes <- function(patributos) {
  atributos <- unique(patributos)
  length(atributos) == length(intersect(atributos, colnames(dataset)))
}
if (atributos_presentes(c("foto_mes")))
  dataset[, kmes := foto_mes %% 100]
if (atributos_presentes(c("mpayroll", "cliente_edad")))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

# ---------- BLOQUE 2: ratios y sumas dentro del mes ----------
# Se calculan ANTES de los rankings: un cociente entre dos montos del mismo mes no tiene inflacion.
# Las sumas de montos se llaman m... (despues se rankean); los cocientes se llaman r_... (no se rankean).
fe_creadas <- character(); fe_salteadas <- character()
# suma por fila que da NA solo si todos los sumandos son NA (no revive los ceros de 202006)
suma_na <- function(...) {
  m <- cbind(...)
  s <- rowSums(m, na.rm = TRUE)
  s[rowSums(!is.na(m)) == 0] <- NA
  s
}
# crea la variable solo si estan todas las columnas que necesita; si no, la saltea y avisa
agregar <- function(nombre, requeridos, expr) {
  if (atributos_presentes(requeridos)) {
    dataset[, (nombre) := eval(expr)]
    fe_creadas <<- c(fe_creadas, nombre)
  } else {
    fe_salteadas <<- c(fe_salteadas, paste0(nombre, " (falta: ",
      paste(setdiff(requeridos, colnames(dataset)), collapse = ", "), ")"))
  }
}

if (PARAM$fe$ratios) {
  # actividad total con tarjetas de credito
  agregar("mtarjetas_consumo", c("mtarjeta_visa_consumo", "mtarjeta_master_consumo"),
          quote(suma_na(mtarjeta_visa_consumo, mtarjeta_master_consumo)))
  agregar("ctarjetas_transacciones", c("ctarjeta_visa_transacciones", "ctarjeta_master_transacciones"),
          quote(suma_na(ctarjeta_visa_transacciones, ctarjeta_master_transacciones)))
  # cuanto del limite de la tarjeta esta usando
  agregar("r_visa_uso_limite", c("Visa_msaldototal", "Visa_mlimitecompra"),
          quote(Visa_msaldototal / Visa_mlimitecompra))
  agregar("r_master_uso_limite", c("Master_msaldototal", "Master_mlimitecompra"),
          quote(Master_msaldototal / Master_mlimitecompra))
  # alguna tarjeta en proceso de cierre o cerrada (status 6, 7 o 9)
  agregar("tarjeta_cerrando", c("Visa_status", "Master_status"),
          quote(as.integer(Visa_status %in% c(6, 7, 9) | Master_status %in% c(6, 7, 9))))
  # que parte de la operatoria es por homebanking
  agregar("r_digital", c("chomebanking_transacciones", "ccallcenter_transacciones", "ccajas_transacciones"),
          quote(chomebanking_transacciones /
                (chomebanking_transacciones + ccallcenter_transacciones + ccajas_transacciones)))
  # cobra el sueldo en el banco; su delta1 = -1 es "dejo de cobrar el sueldo aca"
  agregar("tiene_payroll", c("mpayroll"), quote(as.integer(mpayroll > 0)))
  # saldo y consumo medidos en sueldos
  agregar("r_saldo_sobre_payroll", c("mcuentas_saldo", "mpayroll"),
          quote(ifelse(mpayroll > 0, mcuentas_saldo / mpayroll, NA_real_)))
  agregar("r_consumo_sobre_payroll", c("mtarjetas_consumo", "mpayroll"),
          quote(ifelse(mpayroll > 0, mtarjetas_consumo / mpayroll, NA_real_)))
  # intensidad de uso: transacciones del trimestre por producto contratado
  agregar("r_trx_por_producto", c("ctrx_quarter", "cproductos"), quote(ctrx_quarter / cproductos))
}

# ---------- BLOQUE 2 bis: segunda tanda de ratios, con columnas del dataset gerencial ----------
if (PARAM$fe$ratios2) {
  # endeudamiento: prestamos personales medidos en sueldos
  agregar("r_prestamos_sobre_payroll", c("mprestamos_personales", "mpayroll"),
          quote(ifelse(mpayroll > 0, mprestamos_personales / mpayroll, NA_real_)))
  # que parte de la operatoria es por homebanking (reemplaza a r_digital, que no se puede crear)
  agregar("r_homebanking_sobre_trx", c("chomebanking_transacciones", "ctrx_quarter"),
          quote(chomebanking_transacciones / ctrx_quarter))
  # llamados al call center en relacion a la operatoria: reclamos o gestiones de baja
  agregar("r_callcenter_sobre_trx", c("ccallcenter_transacciones", "ctrx_quarter"),
          quote(ccallcenter_transacciones / ctrx_quarter))
  # donde tiene la plata: que parte del saldo esta en caja de ahorro
  agregar("r_caja_ahorro_sobre_saldo", c("mcaja_ahorro", "mcuentas_saldo"),
          quote(ifelse(mcuentas_saldo > 0, mcaja_ahorro / mcuentas_saldo, NA_real_)))
  # cuanto de lo que el banco gana con el cliente son comisiones
  agregar("r_comisiones_sobre_rentabilidad", c("mcomisiones", "mrentabilidad"),
          quote(mcomisiones / mrentabilidad))
  # margen total que deja el cliente (activos + pasivos)
  agregar("mmargen_total", c("mactivos_margen", "mpasivos_margen"),
          quote(suma_na(mactivos_margen, mpasivos_margen)))
  # pago minimo total de las tarjetas, y medido en sueldos
  agregar("mpagominimo_total", c("Visa_mpagominimo", "Master_mpagominimo"),
          quote(suma_na(Visa_mpagominimo, Master_mpagominimo)))
  agregar("r_pagominimo_sobre_payroll", c("mpagominimo_total", "mpayroll"),
          quote(ifelse(mpayroll > 0, mpagominimo_total / mpayroll, NA_real_)))
  # la tarjeta Visa es reciente o vieja en relacion a la antiguedad del cliente
  agregar("r_visa_alta_sobre_antiguedad", c("Visa_fechaalta", "cliente_antiguedad"),
          quote(Visa_fechaalta / (cliente_antiguedad * 30)))
  # que parte de su vida lleva en el banco
  agregar("r_antiguedad_sobre_edad", c("cliente_antiguedad", "cliente_edad"),
          quote(cliente_antiguedad / (cliente_edad * 12)))
}

if (PARAM$fe$ratios || PARAM$fe$ratios2) {
  message("Ratios: creadas ", length(fe_creadas), ": ", paste(fe_creadas, collapse = ", "))
  if (length(fe_salteadas)) message("  salteadas ", length(fe_salteadas), ": ", paste(fe_salteadas, collapse = "; "))
}

# ---------- BLOQUE 1: data drifting, rankings por mes de los montos ----------
# Cada monto se reemplaza por su posicion relativa dentro del mes, con el cero fijo:
# positivos en (0, 1], negativos en [-1, 0), el cero queda en 0 y los NA quedan NA.
cols_monetarias <- character()
if (PARAM$fe$rankings) {
  rank_cero_fijo <- function(x) {
    r <- rep(NA_real_, length(x))
    pos <- which(x > 0); neg <- which(x < 0)
    r[which(x == 0)] <- 0
    if (length(pos)) r[pos] <- frank(x[pos], ties.method = "average") / length(pos)
    if (length(neg)) r[neg] <- -frank(-x[neg], ties.method = "average") / length(neg)
    r
  }
  cols_monetarias <- grep("^(m|Visa_m|Master_m)", colnames(dataset), value = TRUE)
  # paso todo a numerico antes: una columna entera truncaria los rankings a 0
  for (v in cols_monetarias) set(dataset, j = v, value = as.numeric(dataset[[v]]))
  dataset[, (cols_monetarias) := lapply(.SD, rank_cero_fijo), by = foto_mes, .SDcols = cols_monetarias]
  message("Bloque rankings: ", length(cols_monetarias), " variables de monto convertidas a ranking por mes")
}

# ---------- 6.3.1.5 FE historico: lags y deltas de orden 1 y 2 (identico al z719) ----------
# los lags usan la fila anterior de cada cliente: verifico que esten en orden cronologico
stopifnot(dataset[, all(diff(foto_mes) > 0), by = numero_de_cliente][, all(V1)])

cols_lagueables <- copy(setdiff(colnames(dataset), c("numero_de_cliente", "foto_mes", "clase_ternaria")))
dataset[, paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
        by = numero_de_cliente, .SDcols = cols_lagueables]
dataset[, paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
        by = numero_de_cliente, .SDcols = cols_lagueables]
for (vcol in cols_lagueables) {
  dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
  dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}

# ---------- BLOQUE 3: tendencias en ventana, sobre las variables mas importantes ----------
cols_top <- character()
if (PARAM$fe$tendencias) {
  n <- PARAM$fe$tendencias_ventana

  # lista de variables: se fija una sola vez y queda guardada en la carpeta del experimento (reproducible)
  archivo_cols <- file.path(carpeta_exp, "tendencias_cols.txt")
  if (file.exists(archivo_cols)) {
    cols_top <- readLines(archivo_cols)
  } else {
    # si otro experimento ya fijo la lista (por ejemplo el WF7192), uso exactamente la misma
    previas <- file.path(base, paste0("WF", PARAM$comparar_con), "tendencias_cols.txt")
    previas <- previas[file.exists(previas)]
    candidatos <- file.path(base, c("WF7191/V3gan_desde202103", "WF7191_test/V3gan_desde202103",
                                    "WF7190/V3_desde202103"), "impo.txt")
    origen <- candidatos[file.exists(candidatos)][1]
    if (length(previas)) {
      file.copy(previas[length(previas)], archivo_cols)
      message("Variables para tendencias copiadas de ", previas[length(previas)])
      cols_top <- readLines(archivo_cols)
    } else {
    if (is.na(origen))
      stop("No encuentro el impo.txt del WF7191 ni del WF7190 para elegir las variables de las tendencias. ",
           "Copialo a una de esas carpetas o poné PARAM$fe$tendencias <- FALSE")
    impo <- fread(origen)                                   # viene ordenado por importancia (Gain)
    base_vars <- unique(sub("_(lag|delta)[12]$", "", impo$Feature))
    cols_top <- head(intersect(base_vars, setdiff(cols_lagueables, "kmes")), PARAM$fe$tendencias_top)
    writeLines(cols_top, archivo_cols)
    message("Variables para tendencias tomadas de ", origen)
    }
  }
  cols_top <- intersect(cols_top, cols_lagueables)
  if (!length(cols_top)) stop("La lista de variables para tendencias quedo vacia: revisar ", archivo_cols)

  # trabajo sobre una copia ordenada por cliente y mes, sin cambiar el orden de las filas del dataset
  ord <- order(dataset$numero_de_cliente, dataset$foto_mes)
  pos <- sequence(rle(dataset$numero_de_cliente[ord])$lengths)   # numero de mes dentro de cada cliente
  incompleta <- pos < n                                          # la ventana mezclaria dos clientes
  w <- (1:n) - (n + 1) / 2                                       # pesos de la pendiente de minimos cuadrados
  den <- sum(w^2)
  guardar <- function(nombre, valores) {
    valores[incompleta] <- NA
    res <- rep(NA_real_, length(ord))
    res[ord] <- valores
    set(dataset, j = nombre, value = res)
  }
  for (v in cols_top) {
    x <- as.numeric(dataset[[v]][ord])
    pendiente <- Reduce(`+`, lapply(1:n, function(j) w[j] * shift(x, n - j))) / den
    promedio <- frollmean(x, n)
    guardar(paste0(v, "_tend", n), pendiente)                    # pendiente de los ultimos n meses
    guardar(paste0(v, "_avg", n), promedio)                      # promedio
    guardar(paste0(v, "_max", n), frollapply(x, n, max))         # maximo
    guardar(paste0(v, "_min", n), frollapply(x, n, min))         # minimo
    guardar(paste0(v, "_ratioavg", n), x / promedio)             # valor actual contra su promedio
  }
  message("Bloque tendencias: ", length(cols_top), " variables x 5 = ", 5 * length(cols_top),
          " columnas nuevas (ventana de ", n, " meses)")
}

# ---------- clase binaria, azar y variables del modelo ----------
dataset[, clase01 := ifelse(clase_ternaria %in% PARAM$trainingstrategy$positivos, 1L, 0L)]
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar := runif(nrow(dataset))]

excluir <- c("clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")
if (PARAM$fe$sacar_ids)   # siguen en el dataset para los lags, los filtros y el archivo de Kaggle
  excluir <- c(excluir, "numero_de_cliente", "foto_mes", grep("^kmes", colnames(dataset), value = TRUE))
campos_buenos <- copy(setdiff(colnames(dataset), excluir))

# los cocientes pueden dar Inf o NaN (division por cero): los paso a NA, que LightGBM maneja bien
no_finitos <- 0
for (j in campos_buenos) if (is.double(dataset[[j]])) {
  malos <- which(is.nan(dataset[[j]]) | is.infinite(dataset[[j]]))
  if (length(malos)) { set(dataset, i = malos, j = j, value = NA_real_); no_finitos <- no_finitos + length(malos) }
}

# ---------- CANARITOS: seleccion de variables ----------
# Se agregan variables de puro azar (canaritos) y se entrena con los meses de training, sin mirar julio.
# Si el modelo usa una variable real menos que a los canaritos, esa variable no aporta nada y se descarta.
canaritos_out <- NULL
if (PARAM$fe$canaritos) {
  archivo_can <- file.path(carpeta_exp, paste0("canaritos_", etiqueta, ".csv"))
  if (file.exists(archivo_can)) {
    sel <- fread(archivo_can)
    if (!setequal(sel[es_canarito == FALSE, Feature], campos_buenos))
      stop("La seleccion guardada en ", archivo_can, " es de otro conjunto de variables. Cambia PARAM$experimento.")
    message("Canaritos: seleccion leida de ", archivo_can)
  } else {
    nombres_can <- paste0("canarito", seq_len(PARAM$fe$canaritos_cantidad))
    set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
    for (v in nombres_can) set(dataset, j = v, value = runif(nrow(dataset)))

    meses_can <- sort(unique(dataset$foto_mes))
    meses_can <- meses_can[meses_can >= PARAM$variantes[[1]]$desde & meses_can <= PARAM$trainingstrategy$hasta]
    filtro_can <- dataset[, foto_mes %in% meses_can]
    dcan <- lgb.Dataset(data = data.matrix(dataset[filtro_can, c(campos_buenos, nombres_can), with = FALSE]),
                        label = dataset[filtro_can, clase01], free_raw_data = TRUE)
    param_can <- copy(PARAM$lgbm$param_fijos)
    param_can$early_stopping_rounds <- NULL
    param_can$num_iterations <- PARAM$fe$canaritos_iteraciones
    semillas_can <- PARAM$semillerio$semillas[seq_len(min(PARAM$fe$canaritos_semillas, length(PARAM$semillerio$semillas)))]
    impo_can <- rbindlist(lapply(semillas_can, function(s) {
      param_can$seed <- s
      modelo <- lgb.train(data = dcan, param = param_can, verbose = -100)
      as.data.table(lgb.importance(modelo))[, .(Feature, Gain)]
    }))
    dataset[, (nombres_can) := NULL]
    rm(dcan); invisible(gc(full = TRUE, verbose = FALSE))

    # importancia promedio entre semillas; una variable que un modelo no uso cuenta como 0
    sel <- data.table(Feature = c(campos_buenos, nombres_can))
    sel <- merge(sel, impo_can[, .(gain = sum(Gain) / length(semillas_can)), by = Feature], by = "Feature", all.x = TRUE)
    sel[is.na(gain), gain := 0]
    sel[, es_canarito := Feature %in% nombres_can]
    setorder(sel, -gain)
    sel[, puesto := .I]
    umbral <- sel[es_canarito == TRUE, quantile(gain, PARAM$fe$canaritos_umbral)]
    sel[, queda := !es_canarito & gain > umbral]
    fwrite(sel, archivo_can)
  }

  n_antes <- length(campos_buenos)
  campos_buenos <- campos_buenos[campos_buenos %in% sel[queda == TRUE, Feature]]
  if (length(campos_buenos) < 20)
    stop("Los canaritos dejaron solo ", length(campos_buenos), " variables: revisar PARAM$fe$canaritos_umbral")
  can <- sel[es_canarito == TRUE]
  message("Canaritos: quedan ", length(campos_buenos), " de ", n_antes, " variables (se descartan ",
          n_antes - length(campos_buenos), "). El mejor canarito quedo en el puesto ", min(can$puesto),
          " de ", nrow(sel), "; el canarito mediano, en el ", round(median(can$puesto)))
  message("  Cuantas quedarian con otros umbrales: ",
          paste(sapply(c(0.25, 0.5, 0.75, 0.9, 1), function(q)
            paste0(q, " -> ", sel[es_canarito == FALSE & gain > quantile(can$gain, q), .N])), collapse = ";  "))
  nuevas <- intersect(fe_creadas, sel$Feature)
  if (length(nuevas))
    message("  Ratios (variable del mes): quedan ", paste(intersect(nuevas, campos_buenos), collapse = ", "),
            if (length(setdiff(nuevas, campos_buenos))) paste0(";  se descartan ", paste(setdiff(nuevas, campos_buenos), collapse = ", ")) else "")
  canaritos_out <- list(antes = n_antes, quedan = length(campos_buenos), mejor_canarito_puesto = min(can$puesto),
                        umbral = PARAM$fe$canaritos_umbral)
}

PARAM$fe$out <- list(ratios_creadas = fe_creadas, ratios_salteadas = fe_salteadas,
                     rankeadas = length(cols_monetarias), tendencias_cols = cols_top,
                     canaritos = canaritos_out, variables_modelo = length(campos_buenos))
message(format(Sys.time(), "%a %b %d %X %Y"), "  preprocesado: ", nrow(dataset), " filas x ",
        length(campos_buenos), " variables del modelo  (", no_finitos, " valores no finitos pasados a NA)")

## Funciones: ganancia, semillerío y Kaggle

In [ ]:
# ---------- Kaggle ----------
preparar_kaggle <- function() {
  if (!file.exists("~/.kaggle/kaggle.json"))
    stop("falta ~/.kaggle/kaggle.json en la VM")
  if (Sys.which("kaggle") == "")
    invisible(system("python3 -m pip install -q --user kaggle"))
}

# sube un archivo; si falla (cupo diario, red) lo anota en pendientes_kaggle.txt para reintentar.
# los que ya subieron quedan en subidos_kaggle.txt y no se vuelven a subir al re-correr
# si una subida falla, el resto de la corrida ya no intenta: todo queda pendiente para la proxima vez
kaggle_sin_cupo <- FALSE
subir_a_kaggle <- function(archivo, mensaje) {
  archivo <- normalizePath(archivo)
  if (file.exists(file.path(carpeta_exp, "subidos_kaggle.txt")) &&
      archivo %in% readLines(file.path(carpeta_exp, "subidos_kaggle.txt"))) return(TRUE)
  if (kaggle_sin_cupo) {
    cat(archivo, "|", mensaje, "\n", sep = "", file = file.path(carpeta_exp, "pendientes_kaggle.txt"), append = TRUE)
    return(FALSE)
  }
  salida <- tryCatch(
    suppressWarnings(system2("kaggle", c("competitions", "submit", "-c", PARAM$kaggle$competencia,
                                         "-f", shQuote(archivo), "-m", shQuote(mensaje)),
                             stdout = TRUE, stderr = TRUE)),
    error = function(e) conditionMessage(e))
  ok <- any(grepl("Successfully submitted", salida))
  message("    kaggle ", if (ok) "OK   " else "ERROR", "  ", basename(archivo),
          if (ok) "" else paste0("  ", substr(paste(trimws(salida), collapse = " "), 1, 300)))
  if (ok) {
    cat(archivo, "\n", sep = "", file = file.path(carpeta_exp, "subidos_kaggle.txt"), append = TRUE)
    Sys.sleep(30)
  } else {
    kaggle_sin_cupo <<- TRUE
    cat(archivo, "|", mensaje, "\n", sep = "", file = file.path(carpeta_exp, "pendientes_kaggle.txt"), append = TRUE)
  }
  ok
}

reintentar_pendientes <- function() {
  arch <- file.path(carpeta_exp, "pendientes_kaggle.txt")
  if (!subir_kaggle || !file.exists(arch)) return(invisible())
  pendientes <- unique(setdiff(trimws(readLines(arch)), ""))
  file.remove(arch)
  message(length(pendientes), " archivos pendientes de subir a Kaggle")
  for (p in pendientes) {          # si falla uno, los siguientes quedan anotados sin intentar
    partes <- strsplit(p, "|", fixed = TRUE)[[1]]
    subir_a_kaggle(partes[1], partes[2])
  }
}

# ---------- Training strategy ----------
meses_training <- function(desde) {
  meses <- sort(unique(dataset$foto_mes), decreasing = TRUE)
  meses[meses >= desde & meses <= PARAM$trainingstrategy$hasta]
}

lgb_dataset <- function(filtro) {
  lgb.Dataset(data = data.matrix(dataset[filtro, campos_buenos, with = FALSE]),
              label = dataset[filtro, clase01], free_raw_data = TRUE)
}

# ---------- Ganancia ----------
# ordena por probabilidad, acumula la ganancia de contactar a los primeros N y devuelve el maximo
# de la curva suavizada (promedio movil de PARAM$ganancia$ventana clientes) y el corte donde ocurre
ganancia_maxima <- function(prob, gan) {
  orden <- order(prob, decreasing = TRUE)
  curva <- cumsum(gan[orden])
  suave <- frollmean(curva, PARAM$ganancia$ventana, align = "center")
  corte <- which.max(suave)
  list(ganancia = suave[corte], corte = corte)
}

# ---------- 6.3.2.2 Grid search por GANANCIA en validacion (reanudable) ----------
grid_search <- function(variante, carpeta) {
  archivo <- file.path(carpeta, "tb_grid_search.txt")
  training <- meses_training(variante$desde)
  filtro_train <- dataset[, foto_mes %in% training &
    (clase_ternaria %in% PARAM$trainingstrategy$positivos | azar < PARAM$trainingstrategy$training_pct)]

  # combinaciones repetidas: un arbol no puede tener mas hojas que filas / min_data_in_leaf, asi que todo
  # num_leaves por encima de ese tope da exactamente el mismo modelo. Se evalua una sola de cada grupo.
  tb <- CJ(num_leaves = PARAM$grid$num_leaves, min_data_in_leaf = PARAM$grid$min_data_in_leaf)
  total <- nrow(tb)
  tb[, hojas_efectivas := pmin(num_leaves, floor(sum(filtro_train) / min_data_in_leaf))]
  tb <- tb[, .(num_leaves = min(num_leaves)), by = .(hojas_efectivas, min_data_in_leaf)][
    order(num_leaves, min_data_in_leaf), .(num_leaves, min_data_in_leaf)]
  message("  grilla: ", total, " combinaciones, ", nrow(tb), " distintas; ",
          PARAM$grid$semillas_eval, " semillas cada una")
  if (file.exists(archivo)) tb <- tb[!fread(archivo), on = .(num_leaves, min_data_in_leaf)]

  if (nrow(tb)) {
    filtro_valid <- dataset[, foto_mes %in% PARAM$trainingstrategy$validate]
    dtrain <- lgb_dataset(filtro_train)
    dvalidate <- lgb_dataset(filtro_valid)
    mvalid <- data.matrix(dataset[filtro_valid, campos_buenos, with = FALSE])
    # ganancia de contactar a cada cliente de validacion, en el mismo orden que dvalidate
    gan_valid <- dataset[filtro_valid, ifelse(clase_ternaria == "BAJA+2", PARAM$ganancia$acierto, PARAM$ganancia$error)]

    # metrica para el early stopping: la ganancia maxima suavizada en validacion
    feval_ganancia <- function(preds, dtrain) {
      list(name = "ganancia", value = ganancia_maxima(preds, gan_valid)$ganancia, higher_better = TRUE)
    }

    semillas_eval <- PARAM$semillerio$semillas[seq_len(PARAM$grid$semillas_eval)]
    for (i in seq_len(nrow(tb))) {
      x <- as.list(tb[i])
      evals <- rbindlist(lapply(semillas_eval, function(s) {
        param <- modifyList(PARAM$lgbm$param_fijos, c(x, list(seed = s)))
        modelo <- lgb.train(data = dtrain, valids = list(valid = dvalidate), eval = feval_ganancia,
                            param = param, verbose = -100)
        niter <- modelo$best_iter
        g <- ganancia_maxima(predict(modelo, mvalid, num_iteration = niter), gan_valid)
        rm(modelo); gc(full = TRUE, verbose = FALSE)
        data.table(semilla = s, ganancia = g$ganancia, corte = g$corte, num_iterations = niter)
      }))
      fila <- data.table(tb[i], ganancia = mean(evals$ganancia), ganancia_sd = sd(evals$ganancia),
                         corte = round(mean(evals$corte)), num_iterations = round(mean(evals$num_iterations)))
      fwrite(fila, archivo, sep = "\t", append = file.exists(archivo))
      message(format(Sys.time(), "%a %b %d %X %Y  "), variante$nombre, "  nl=", x$num_leaves,
              " mdil=", x$min_data_in_leaf, "  niter ", fila$num_iterations, "  corte ", fila$corte,
              "  ganancia ", round(fila$ganancia / 1e6, 2), " M")
    }
    rm(dtrain, dvalidate, mvalid); gc(full = TRUE, verbose = FALSE)
  }
  res <- fread(archivo)
  setorder(res, -ganancia)
  res
}

# ---------- 6.3.3 Final training con semillerio (reanudable: una prediccion por semilla) ----------
semillerio <- function(variante, carpeta, mejores) {
  carpeta_sem <- file.path(carpeta, "semillas")
  dir.create(carpeta_sem, showWarnings = FALSE)

  fijos <- copy(PARAM$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL
  # modifyList: los mejores del grid PISAN a los fijos (con c() quedaban duplicados y LightGBM usaba los fijos)
  param_final <- modifyList(fijos, mejores)

  pendientes <- PARAM$semillerio$semillas[
    !file.exists(file.path(carpeta_sem, paste0("pred_s", PARAM$semillerio$semillas, ".txt")))]

  final_train <- c(PARAM$trainingstrategy$validate, meses_training(variante$desde))
  if (isTRUE(PARAM$final$escalar_iteraciones)) {
    # el grid eligio las iteraciones con los meses de training; el final_train suma 202107
    factor <- dataset[foto_mes %in% final_train, .N] / dataset[foto_mes %in% meses_training(variante$desde), .N]
    param_final$num_iterations <- round(param_final$num_iterations * factor)
  }

  if (length(pendientes)) {
    dfinal_train <- lgb_dataset(dataset[, foto_mes %in% final_train])
    dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]
    mfuture <- data.matrix(dfuture[, campos_buenos, with = FALSE])

    for (s in pendientes) {
      param_final$seed <- s
      modelo <- lgb.train(data = dfinal_train, param = param_final, verbose = -100)
      fwrite(data.table(numero_de_cliente = dfuture$numero_de_cliente, prob = predict(modelo, mfuture)),
             file.path(carpeta_sem, paste0("pred_s", s, ".txt")), sep = "\t")
      if (s == PARAM$semilla_primigenia) {          # modelo e importancia solo de la semilla primigenia
        lgb.save(modelo, file.path(carpeta, "modelo.txt"))
        fwrite(as.data.table(lgb.importance(modelo)), file.path(carpeta, "impo.txt"), sep = "\t")
      }
      message(format(Sys.time(), "%a %b %d %X %Y  "), variante$nombre, "  semilla ", s, " lista")
      rm(modelo); gc(full = TRUE, verbose = FALSE)
    }
    rm(dfinal_train, dfuture, mfuture); gc(full = TRUE, verbose = FALSE)
  }

  # promedio de las probabilidades de todas las semillas
  preds <- rbindlist(lapply(PARAM$semillerio$semillas, function(s)
    fread(file.path(carpeta_sem, paste0("pred_s", s, ".txt")))[, semilla := s]))
  tb_prediccion <- preds[, .(prob = mean(prob)), by = numero_de_cliente]
  fwrite(tb_prediccion, file.path(carpeta, "prediccion.txt"), sep = "\t")   # util para ensembles
  list(prediccion = tb_prediccion, param_final = param_final)
}

# ---------- Archivos para Kaggle ----------
generar_kaggle <- function(variante, carpeta, tb_prediccion) {
  dir.create(file.path(carpeta, "kaggle"), showWarnings = FALSE)
  tb <- copy(tb_prediccion)
  setorder(tb, -prob)
  for (envios in PARAM$kaggle$cortes) {
    tb[, Predicted := 0L]
    tb[1:envios, Predicted := 1L]
    archivo <- file.path(carpeta, "kaggle",
                         paste0("KA", PARAM$experimento, "_", variante$nombre, "_", envios, ".csv"))
    fwrite(tb[, .(numero_de_cliente, Predicted)], archivo, sep = ",")
    if (subir_kaggle)
      subir_a_kaggle(archivo, paste0("z719 ", variante$nombre, " semillerio=", PARAM$semillerio$cantidad,
                                     " semilla=", PARAM$semilla_primigenia, " envios=", envios))
  }
}

## Comparación: las 8 configuraciones con 40 semillas

In [ ]:
# ===== Barrido de hiperparametros: un parametro por vez, medido en julio =====
# Cada configuracion se evalua con las mismas semillas, entrenando con 202103-202106 y midiendo en 202107.
# Reanudable: cada configuracion terminada queda guardada y no se repite.
variante <- PARAM$variantes[[1]]
carpeta_barrido <- file.path(carpeta_exp, paste0("barrido_", variante$nombre))
dir.create(carpeta_barrido, showWarnings = FALSE)
archivo_semillas <- file.path(carpeta_barrido, "barrido_semillas.csv")
archivo_resumen  <- file.path(carpeta_barrido, "barrido_resumen.csv")
archivo_grupos   <- file.path(carpeta_barrido, "barrido_grupos.csv")
semillas <- PARAM$semillerio$semillas[seq_len(min(PARAM$barrido$semillas, length(PARAM$semillerio$semillas)))]

# la reanudacion solo vale si la configuracion general es la misma
firma <- sub("\n$", "", as.yaml(list(base = PARAM$base, param_fijos = PARAM$lgbm$param_fijos, semillas = semillas,
  ganancia = PARAM$ganancia, training = meses_training(variante$desde), validate = PARAM$trainingstrategy$validate,
  positivos = PARAM$trainingstrategy$positivos, variables_modelo = campos_buenos)))
archivo_firma <- file.path(carpeta_barrido, "config_corrida.yml")
if (file.exists(archivo_resumen) &&
    (!file.exists(archivo_firma) || !identical(paste(readLines(archivo_firma), collapse = "\n"), firma)))
  stop("La carpeta ", carpeta_barrido, " tiene un barrido hecho con OTRA configuracion (base, semillas o variables). ",
       "Cambia PARAM$experimento para correr en una carpeta nueva.")
writeLines(firma, archivo_firma)

# lista de configuraciones: la base, cada parametro movido de a uno, y las extras
configs <- list(base = list())
for (p in names(PARAM$barrido$valores)) for (v in PARAM$barrido$valores[[p]]) {
  cambios <- setNames(list(v), p)
  if (p == "bagging_fraction") cambios$bagging_freq <- 1      # el bagging solo actua si la frecuencia es mayor a 0
  configs[[paste0(p, "=", v)]] <- cambios
}
for (e in names(PARAM$barrido$extras)) configs[[e]] <- PARAM$barrido$extras[[e]]
hechas <- if (file.exists(archivo_resumen)) fread(archivo_resumen)$config else character()
pendientes <- setdiff(names(configs), hechas)
message(length(configs), " configuraciones, ", length(pendientes), " pendientes; ", length(semillas), " semillas cada una")

if (length(pendientes)) {
  filtro_train <- dataset[, foto_mes %in% meses_training(variante$desde) &
    (clase_ternaria %in% PARAM$trainingstrategy$positivos | azar < PARAM$trainingstrategy$training_pct)]
  filtro_valid <- dataset[, foto_mes %in% PARAM$trainingstrategy$validate]
  mtrain <- data.matrix(dataset[filtro_train, campos_buenos, with = FALSE]); ytrain <- dataset[filtro_train, clase01]
  mvalid <- data.matrix(dataset[filtro_valid, campos_buenos, with = FALSE]); yvalid <- dataset[filtro_valid, clase01]
  gan_valid <- dataset[filtro_valid, ifelse(clase_ternaria == "BAJA+2", PARAM$ganancia$acierto, PARAM$ganancia$error)]
  feval_ganancia <- function(preds, dtrain) {
    list(name = "ganancia", value = ganancia_maxima(preds, gan_valid)$ganancia, higher_better = TRUE)
  }

  for (etq in pendientes) {
    t0 <- Sys.time()
    hiper <- modifyList(PARAM$base, configs[[etq]])
    param <- modifyList(PARAM$lgbm$param_fijos, hiper)
    # datasets nuevos para cada configuracion: max_bin queda fijado cuando se construye el dataset
    dtrain <- lgb.Dataset(data = mtrain, label = ytrain, free_raw_data = FALSE)
    dvalid <- lgb.Dataset(data = mvalid, label = yvalid, free_raw_data = FALSE)

    modelos <- lapply(semillas, function(s) {
      param$seed <- s
      lgb.train(data = dtrain, valids = list(valid = dvalid), eval = feval_ganancia, param = param, verbose = -100)
    })
    iters <- sapply(modelos, function(m) m$best_iter)
    iter_comun <- round(mean(iters))      # la cantidad de arboles que usaria el modelo final

    # tres medidas:
    #  gan_es   : cada modelo cortado en su mejor arbol (como en el grid search; optimista)
    #  gan_fija : cada modelo con la cantidad comun de arboles (como se entrena el modelo final)
    #  gan_sem  : el semillerio, promedio de las probabilidades de todos los modelos con arboles fijos
    g_es <- sapply(seq_along(modelos), function(i)
      ganancia_maxima(predict(modelos[[i]], mvalid, num_iteration = iters[i]), gan_valid)$ganancia)
    probs <- sapply(modelos, function(m) predict(m, mvalid, num_iteration = iter_comun))
    g_fija <- apply(probs, 2, function(p) ganancia_maxima(p, gan_valid)$ganancia)
    sem <- ganancia_maxima(rowMeans(probs), gan_valid)

    fwrite(data.table(config = etq, semilla = semillas, iter = iters, gan_es = g_es, gan_fija = g_fija),
           archivo_semillas, append = file.exists(archivo_semillas))
    # semillerios de a PARAM$barrido$grupo semillas, con grupos disjuntos e iguales en todas las configuraciones
    tam <- PARAM$barrido$grupo
    if (!is.null(tam) && length(semillas) >= 2 * tam) {
      grupos <- split(seq_along(semillas), ceiling(seq_along(semillas) / tam))
      grupos <- grupos[lengths(grupos) == tam]
      fwrite(rbindlist(lapply(seq_along(grupos), function(g) {
        r <- ganancia_maxima(rowMeans(probs[, grupos[[g]], drop = FALSE]), gan_valid)
        data.table(config = etq, grupo = g, semillas = tam, gan_sem = r$ganancia, corte = r$corte)
      })), archivo_grupos, append = file.exists(archivo_grupos))
    }
    fwrite(data.table(config = etq, hiperparametros = paste(names(hiper), unlist(hiper), sep = "=", collapse = " "),
                      iter_comun = iter_comun, gan_es = mean(g_es), gan_fija = mean(g_fija), gan_fija_sd = sd(g_fija),
                      gan_sem = sem$ganancia, corte_sem = sem$corte,
                      minutos = round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1)),
           archivo_resumen, append = file.exists(archivo_resumen))
    message(format(Sys.time(), "%a %b %d %X %Y  "), sprintf("%-24s", etq), " arboles ", iter_comun,
            "  individual ", round(mean(g_fija) / 1e6, 2), " M  semillerio ", round(sem$ganancia / 1e6, 2),
            " M  corte ", sem$corte)
    rm(modelos, dtrain, dvalid, probs); invisible(gc(full = TRUE, verbose = FALSE))
  }
  rm(mtrain, mvalid); invisible(gc(full = TRUE, verbose = FALSE))
}
message(format(Sys.time(), "%a %b %d %X %Y"), "  barrido completo")

## Resultado
Cada configuración contra la base; el efecto del learning rate combinación por combinación; el efecto promedio; y el veredicto según la regla.

In [ ]:
# ===== Resultado del barrido: cada configuracion contra la base, de a pares por semilla =====
res <- fread(archivo_resumen)
sem <- fread(archivo_semillas)
base_sem <- sem[config == "base", .(semilla, base_fija = gan_fija)]
pares <- merge(sem[config != "base"], base_sem, by = "semilla")
dif <- pares[, .(dif_M = mean(gan_fija - base_fija) / 1e6,
                 error_M = sd(gan_fija - base_fija) / sqrt(.N) / 1e6,
                 semillas_mejor = sum(gan_fija > base_fija), n = .N), by = config]
tabla <- merge(res, dif, by = "config", all.x = TRUE)
tabla[, parametro := fifelse(grepl("=", config), sub("=.*", "", config), config)]
tabla[, valor := suppressWarnings(as.numeric(sub(".*=", "", config)))]
tabla[, errores := fifelse(!is.na(error_M) & error_M > 0, dif_M / error_M, 0)]
tabla[, veredicto := fifelse(config == "base", "base",
                     fifelse(abs(errores) < 2, "igual a la base", fifelse(dif_M > 0, "MEJOR", "peor")))]
tabla[, `:=`(individual_M = round(gan_fija / 1e6, 2), semillerio_M = round(gan_sem / 1e6, 2),
             dif_sem_M = round((gan_sem - gan_sem[config == "base"]) / 1e6, 2))]
setorder(tabla, parametro, valor)
fwrite(tabla, file.path(carpeta_barrido, "barrido_tabla.csv"))

fila_base <- tabla[config == "base"]
message("BASE: ", fila_base$hiperparametros, "  |  individual ", fila_base$individual_M, " M, semillerio ",
        fila_base$semillerio_M, " M, ", fila_base$iter_comun, " arboles, corte ", fila_base$corte_sem)
message("dif_M = diferencia de la ganancia individual contra la base, con las mismas semillas. ",
        "errores = dif_M / su error: por debajo de 2 no se distingue de la base.\n")
cols_tabla <- function(x) x[, .(config, arboles = iter_comun, individual_M, dif_M = round(dif_M, 2),
                                error_M = round(error_M, 2), errores = round(errores, 1),
                                semillas_mejor = paste0(semillas_mejor, "/", n), semillerio_M, dif_sem_M,
                                corte = corte_sem, veredicto)]
for (p in unique(tabla[grepl("=", config), parametro])) {
  cat("\n---", p, "(sobre la base) ---\n")
  print(cols_tabla(tabla[parametro == p]), row.names = FALSE)
}
if (tabla[!grepl("=", config) & config != "base", .N]) {
  cat("\n--- otras combinaciones de num_leaves y min_data_in_leaf (cada una contra la base) ---\n")
  print(cols_tabla(tabla[!grepl("=", config) & config != "base"][order(config)]), row.names = FALSE)
}

# cuanto mueve cada parametro: rango de la ganancia individual entre sus valores (incluida la base)
sens <- tabla[grepl("=", config), .(valores = .N, mejor_dif_M = round(max(dif_M), 2), peor_dif_M = round(min(dif_M), 2),
                                    rango_M = round(max(c(dif_M, 0)) - min(c(dif_M, 0)), 2),
                                    alguno_mejor = any(veredicto == "MEJOR")), by = parametro][order(-rango_M)]
cat("\n=== Cuanto mueve cada parametro (millones, contra la base) ===\n"); print(sens, row.names = FALSE)
fwrite(sens, file.path(carpeta_barrido, "barrido_sensibilidad.csv"))

# grafico: un panel por parametro, diferencia contra la base con barras de +- 2 errores
dibujar_barrido <- function() {
  ps <- setdiff(unique(tabla[grepl("=", config), parametro]), "base")
  filas <- ceiling(length(ps) / 4)
  op <- par(mfrow = c(filas, 4), mar = c(4, 4, 2.5, 1)); on.exit(par(op))
  ylim <- range(c(tabla$dif_M - 2 * tabla$error_M, tabla$dif_M + 2 * tabla$error_M, 0), na.rm = TRUE)
  for (p in ps) {
    x <- tabla[parametro == p]
    plot(seq_len(nrow(x)), x$dif_M, ylim = ylim, xaxt = "n", pch = 19, col = "#1F6FB2", cex = 1.3,
         xlab = p, ylab = "Diferencia contra la base (M)", main = p, xlim = c(0.5, nrow(x) + 0.5))
    axis(1, at = seq_len(nrow(x)), labels = x$valor)
    abline(h = 0, col = "#C77700", lwd = 2)
    suppressWarnings(arrows(seq_len(nrow(x)), x$dif_M - 2 * x$error_M, seq_len(nrow(x)), x$dif_M + 2 * x$error_M,
           angle = 90, code = 3, length = 0.04, col = "#1F6FB2"))
  }
}
png(file.path(carpeta_barrido, "barrido_grafico.png"), width = 1700, height = 950, res = 130); dibujar_barrido(); invisible(dev.off())
dibujar_barrido()

# semillerios de a pocas semillas: misma comparacion, pero con lo que realmente se sube a Kaggle
if (file.exists(archivo_grupos)) {
  gr <- fread(archivo_grupos)
  gb <- gr[config == "base", .(grupo, base = gan_sem)]
  gp <- merge(gr[config != "base"], gb, by = "grupo")
  cat("\n=== Semillerios de a", gr$semillas[1], "semillas (millones), grupo por grupo ===\n")
  print(dcast(gr, grupo ~ config, value.var = "gan_sem", fun.aggregate = function(x) round(x[1] / 1e6, 2)), row.names = FALSE)
  print(gp[, .(grupos = .N, dif_media_M = round(mean(gan_sem - base) / 1e6, 2),
               error_M = round(sd(gan_sem - base) / sqrt(.N) / 1e6, 2), grupos_mejor = sum(gan_sem > base)), by = config],
        row.names = FALSE)
  message("Semillerio con TODAS las semillas: ", paste0(tabla$config, " ", tabla$semillerio_M, " M", collapse = "  |  "))
}

# ===== learning_rate 0.02 contra 0.03 DENTRO de cada combinacion de num_leaves y min_data_in_leaf =====
nombres_03 <- grep("_lr0\\.03$", names(configs), value = TRUE)
pares_lr <- c(list(c("base", "learning_rate=0.02")), lapply(nombres_03, function(x) c(x, sub("_lr0\\.03$", "_lr0.02", x))))
pares_lr <- pares_lr[sapply(pares_lr, function(p) all(p %in% sem$config))]
juntar <- function(p) merge(sem[config == p[1], .(semilla, a = gan_fija)], sem[config == p[2], .(semilla, b = gan_fija)], by = "semilla")
comp <- rbindlist(lapply(pares_lr, function(p) {
  m <- juntar(p); d <- (m$b - m$a) / 1e6
  data.table(combinacion = sub("_lr0\\.03$", "", sub("^base$", "nl256_mdil128 (base)", p[1])),
             individual_lr0.03 = round(mean(m$a) / 1e6, 2), individual_lr0.02 = round(mean(m$b) / 1e6, 2),
             dif_M = round(mean(d), 2), error_M = round(sd(d) / sqrt(length(d)), 2),
             errores = round(mean(d) / (sd(d) / sqrt(length(d))), 1), semillas_mejor = paste0(sum(d > 0), "/", length(d)),
             semillerio_lr0.03 = tabla[config == p[1], semillerio_M], semillerio_lr0.02 = tabla[config == p[2], semillerio_M])
}))
cat("\n=== Efecto de bajar learning_rate de 0.03 a 0.02, combinacion por combinacion (millones) ===\n")
print(comp, row.names = FALSE)
fwrite(comp, file.path(carpeta_barrido, "barrido_learning_rate.csv"))

# efecto promedio: para cada semilla, el promedio de la diferencia en todas las combinaciones
por_semilla <- rbindlist(lapply(pares_lr, juntar))[, .(d = mean(b - a) / 1e6), by = semilla]
efecto <- mean(por_semilla$d); efecto_error <- sd(por_semilla$d) / sqrt(nrow(por_semilla))
message("\nEfecto promedio en las ", length(pares_lr), " combinaciones: ", round(efecto, 2), " M  (error ",
        round(efecto_error, 2), " M, es decir ", round(efecto / efecto_error, 1), " errores; mejor en ",
        sum(por_semilla$d > 0), " de ", nrow(por_semilla), " semillas; positivo en ", sum(comp$dif_M > 0), " de ",
        nrow(comp), " combinaciones)")

# regla fijada en los parametros: decide por el efecto promedio, no por la mejor configuracion suelta
cumple <- efecto > PARAM$barrido$umbral_M && efecto / efecto_error > 2 && sum(comp$dif_M > 0) >= min(3, nrow(comp))
candidatas <- if (cumple) tabla[config == "learning_rate=0.02"] else tabla[0]
if (cumple) {
  message("CUMPLE la regla: bajar el learning rate a 0.02 mejora. Modelo final: learning_rate = 0.02 sobre la base.")
} else {
  message("NO cumple la regla (mas de ", PARAM$barrido$umbral_M, " M, mas de 2 errores y positivo en al menos 3 ",
          "combinaciones): el learning rate queda en 0.03 y la entrega sigue siendo la del WF7193.")
}

## Modelo final (solo si se cumple la regla)

In [ ]:
# ===== Modelo final con la mejor configuracion del barrido (solo si supera la regla) =====
# Entrena el semillerio de 20 con 202103-202107, predice 202109 y sube los cortes a Kaggle.
# Si ninguna configuracion supera a la base, no hace nada: la entrega sigue siendo la del WF7193.
elegida <- if (nzchar(PARAM$barrido$final_manual)) PARAM$barrido$final_manual else
           if (nrow(candidatas)) candidatas$config[1] else NA_character_

if (is.na(elegida)) {
  message("No se entrena un modelo final: ninguna configuracion paso la regla. ",
          "Para forzar una, poner su etiqueta en PARAM$barrido$final_manual.")
} else {
  if (!elegida %in% tabla$config) stop("No existe la configuracion '", elegida, "' en el barrido")
  hiper <- modifyList(PARAM$base, configs[[elegida]])
  mejores <- c(hiper, list(num_iterations = tabla[config == elegida, iter_comun]))
  variante_final <- list(nombre = paste0(variante$nombre, "_", gsub("[^A-Za-z0-9._]", "", elegida)),
                         desde = variante$desde,
                         descripcion = paste0("configuracion '", elegida, "' del barrido sobre ", variante$nombre))
  carpeta <- file.path(carpeta_exp, variante_final$nombre)
  dir.create(carpeta, showWarnings = FALSE)
  message("Modelo final con '", elegida, "': ", paste(names(mejores), unlist(mejores), sep = "=", collapse = " "))

  if (subir_kaggle) { preparar_kaggle(); reintentar_pendientes() }
  fin <- semillerio(variante_final, carpeta, mejores)
  generar_kaggle(variante_final, carpeta, fin$prediccion)

  PARAM_var <- PARAM
  PARAM_var$variante <- variante_final
  PARAM_var$trainingstrategy$training <- meses_training(variante_final$desde)
  PARAM_var$trainingstrategy$final_train <- c(PARAM$trainingstrategy$validate, meses_training(variante_final$desde))
  PARAM_var$out$barrido <- as.list(tabla[config == elegida, .(config, hiperparametros, iter_comun, individual_M,
                                                              dif_M, error_M, semillerio_M, corte_sem)])
  PARAM_var$out$param_final <- fin$param_final
  write_yaml(PARAM_var, file.path(carpeta, "PARAM.yml"))
  reintentar_pendientes()
  message(format(Sys.time(), "%a %b %d %X %Y"), "  listo: ", variante_final$nombre)
}

## Puntajes de Kaggle y elección del corte
Solo tiene envíos nuevos si la celda anterior entrenó un modelo final.

In [ ]:
descargar_envios <- TRUE   # FALSE para reusar el envios_kaggle.csv ya bajado

if (descargar_envios) {
  preparar_kaggle()
  # la CLI lista hasta 200 envios: bajo la lista completa con la API de Python, pagina por pagina
  writeLines(c(
    "import csv, sys",
    "from kaggle.api.kaggle_api_extended import KaggleApi",
    "from kagglesdk.competitions.types.competition_api_service import ApiListSubmissionsRequest",
    "from kagglesdk.competitions.types.competition_enums import SubmissionGroup, SubmissionSortBy",
    "api = KaggleApi()",
    "api.authenticate()",
    "envios, token = [], ''",
    "for pagina in range(50):",
    "    with api.build_kaggle_client() as kaggle:",
    "        req = ApiListSubmissionsRequest()",
    "        req.competition_name = sys.argv[1]",
    "        req.group = SubmissionGroup.SUBMISSION_GROUP_ALL",
    "        req.sort_by = SubmissionSortBy.SUBMISSION_SORT_BY_DATE",
    "        api._set_paging(req, 200, token, -1)",
    "        resp = kaggle.competitions.competition_api_client.list_submissions(req)",
    "    envios.extend(resp.submissions or [])",
    "    token = getattr(resp, 'next_page_token', '') or ''",
    "    if not token or not resp.submissions:",
    "        break",
    "with open(sys.argv[2], 'w', newline='', encoding='utf-8') as f:",
    "    w = csv.writer(f)",
    "    w.writerow(['ref', 'archivo', 'descripcion', 'fecha', 'estado', 'public'])",
    "    for s in envios:",
    "        w.writerow([s.ref, s.file_name, s.description, s.date, s.status, s.public_score])",
    "print(len(envios), 'envios en la competencia')"
  ), "bajar_envios_kaggle.py")
  invisible(system(paste("python3 bajar_envios_kaggle.py", PARAM$kaggle$competencia, "envios_kaggle.csv")))
}

# envios de este notebook y de los experimentos a comparar: KA<experimento>_<variante>_<envios>.csv
patron <- paste0("^KA(", paste(c(PARAM$experimento, PARAM$comparar_con), collapse = "|"), ")_(.+)_([0-9]+)\\.csv$")
envios_kaggle <- fread("envios_kaggle.csv", colClasses = "character")
envios_kaggle[, public := suppressWarnings(as.numeric(public))]
puntajes <- envios_kaggle[grepl(patron, archivo) & grepl("COMPLETE", estado) & !is.na(public)]
puntajes[, variante := paste0(sub(patron, "\\1", archivo), "_", sub(patron, "\\2", archivo))]
puntajes[, envios := as.integer(sub(patron, "\\3", archivo))]
setorder(puntajes, fecha)
puntajes <- puntajes[, .SD[.N], by = .(variante, envios)]   # si se subio dos veces, el mas reciente
fwrite(puntajes[, .(variante, envios, public, fecha, ref, archivo)], "puntajes_kaggle.csv")

# curva de ganancia Public por corte
curva <- dcast(puntajes, envios ~ variante, value.var = "public")
print(curva)

# el pico aislado del Public es ruido: elijo el corte con mejor promedio en una ventana de 3 cortes vecinos
setorder(puntajes, variante, envios)
puntajes[, ventana3 := frollmean(public, 3, align = "center"), by = variante]
mejor <- puntajes[!is.na(ventana3)][order(-ventana3)][1]
fwrite(puntajes, "puntajes_ventanas.csv")
message("Maximo Public: ", puntajes[which.max(public), paste0(variante, " en ", envios, " envios (", public, ")")])
message("Corte estable sugerido: ", mejor$variante, " en ", mejor$envios, " envios",
        "  (promedio de la ventana ", round(mejor$ventana3, 3), ", Public del corte ", mejor$public, ")")
message("Archivo a marcar como entrega final en Kaggle: ", mejor$archivo)

## Paquete con los resultados para analizar
Deja `para_analizar_7197.tar.gz` en la carpeta `exp`.

In [ ]:
# ===== Paquete con los resultados para analizar =====
# Requiere las celdas 1 y 2. Junta los archivos chicos de todas las corridas en un solo comprimido.
# No incluye el dataset ni las predicciones por cliente.
experimentos <- unique(c(PARAM$comparar_con, PARAM$experimento))
patrones <- c("^tb_grid_search.*\\.txt$", "^impo\\.txt$", "^PARAM\\.yml$", "^config_corrida\\.yml$",
              "^resumen_variantes\\.csv$", "^comparacion_validacion\\.csv$", "^puntajes_kaggle\\.csv$",
              "^puntajes_ventanas\\.csv$", "^tendencias_cols\\.txt$", "^canaritos_.*\\.csv$",
              "^curva_julio\\.csv$", "^barrido_.*\\.(csv|png)$", "^julio_vs_public\\.csv$", "^curva_julio_vs_public\\.png$")
nombre_paquete <- paste0("para_analizar_", PARAM$experimento)
destino <- file.path(base, nombre_paquete)
unlink(destino, recursive = TRUE)
dir.create(destino, showWarnings = FALSE)
for (carp in file.path(base, paste0("WF", experimentos))) {
  encontrados <- list.files(carp, recursive = TRUE, full.names = TRUE)
  encontrados <- encontrados[grepl(paste(patrones, collapse = "|"), basename(encontrados))]
  for (f in encontrados) {
    relativo <- gsub("/", "__", substring(f, nchar(base) + 2))
    file.copy(f, file.path(destino, relativo), overwrite = TRUE)
  }
}
fwrite(dataset[, .N, by = .(foto_mes, clase_ternaria)][order(foto_mes, clase_ternaria)],
       file.path(destino, "clases_por_mes.csv"))
writeLines(colnames(dataset), file.path(destino, "columnas_dataset.txt"))
writeLines(campos_buenos, file.path(destino, "variables_modelo.txt"))
anterior <- setwd(base)
tar(paste0(nombre_paquete, ".tar.gz"), nombre_paquete, compression = "gzip")
setwd(anterior)
message("Listo: ", file.path(base, paste0(nombre_paquete, ".tar.gz")), "  (", length(list.files(destino)),
        " archivos, ", round(file.size(file.path(base, paste0(nombre_paquete, ".tar.gz"))) / 1024), " KB)")